In [2]:
import os

import healpy as hp
import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.example_libraries import stax

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Predictive, Trace_ELBO, TraceGraph_ELBO, RenyiELBO, autoguide
from numpyro.infer.initialization import init_to_median, init_to_uniform
from numpyro.infer.reparam import NeuTraReparam
from numpyro.infer import MCMC, NUTS
from numpyro import optim
from numpyro import handlers
from numpyro.contrib.tfp.mcmc import ReplicaExchangeMC
from tensorflow_probability.substrates import jax as tfp

import optax
from einops import repeat

from fpp.models.svi import run_svi_with_beta
from fpp.models.scd import dnds
from fpp.models.templates import NFWTemplate, LorimerDiskTemplate
from fpp.models.bulge_models import BulgeTemplates
from fpp.likelihoods.npll_jax import log_like_np
from fpp.utils.sph_harm import Ylm
from fpp.utils import create_mask as cm
from fpp.utils.utils import jnp_trapezoid
from fpp.models.psf import KingPSF
from fpp.utils.psf_correction import PSFCorrection
from fpp.simulations.simulator import simulator

In [1]:
from fpp.models.np_model import NPModel

In [4]:
m = NPModel()

No data provided. Using Fermi data.


In [ ]:
self = m

def model(data=None, beta=1.):
    """Main numpyro model."""

    mu = jnp.zeros_like(data)

    #=== diffuse: pib with spherical harmonics, ics ===
    theta_pib = numpyro.sample("theta_pib", dist.Dirichlet(jnp.ones((self.n_dif,)) / self.n_dif))
    temp_pib = jnp.sum(theta_pib[:, None] * self.pib, 0)
    theta_ics = numpyro.sample("theta_ics", dist.Dirichlet(jnp.ones((self.n_dif,)) / self.n_dif))
    temp_ics = jnp.sum(theta_ics[:, None] * self.ics, 0)

    pib_modifier = jnp.zeros_like(data)
    for i in range(len(self.Ylm_temps)):
        Alm = numpyro.sample(f'Alm_{i}', dist.Uniform(-0.05, 0.05))
        pib_modifier += Alm * self.Ylm_temps[i]
    temp_pib = (1 + pib_modifier) * temp_pib
    temp_pib /= jnp.mean(temp_pib[~self.nm]) # re-normalize after modulation

    mu += numpyro.sample("S_pib", dist.Uniform(1e-3, 14)) * temp_pib
    mu += numpyro.sample("S_ics", dist.Uniform(1e-3, 14)) * temp_ics

    #=== diffuse: isotropic, fermi bubble, (resolved) point sources ===
    mu += numpyro.sample("S_iso", dist.Uniform(1e-3, 5.)) * self.temp_iso
    mu += numpyro.sample("S_bub", dist.Uniform(1e-3, 5.)) * self.temp_bub
    mu += numpyro.sample("S_psc", dist.Uniform(1e-3, 5.)) * self.temp_psc

    #=== diffuse: gce (defined as bulge + nfw) ===
    S_gce = numpyro.sample("S_gce", dist.Uniform(1e-5, 4.))
    f_bulge_poiss = numpyro.sample("f_bulge_poiss", dist.Uniform(0., 1.))

    theta_blg_poiss = numpyro.sample("theta_bulge_poiss", dist.Dirichlet(jnp.ones((self.n_blg,)) / self.n_blg))
    temp_blg_poiss = jnp.sum(theta_blg_poiss[:, None] * self.blg_s, 0)

    temp_nfw_poiss = self.nfw_temp_gen.get_NFW2_template(gamma=numpyro.sample("gamma_poiss", dist.Uniform(0.2, 2.)))
    temp_nfw_poiss /= jnp.mean(temp_nfw_poiss[~self.nm])

    mu += S_gce * (f_bulge_poiss * temp_blg_poiss + (1 - f_bulge_poiss) * temp_nfw_poiss)
                                        
    #=== point source: gce (defined as bulge + nfw) ===
    Sps_gce = numpyro.sample("Sps_gce", dist.Uniform(1e-5, 4.))
    f_bulge_ps = numpyro.sample("f_bulge_ps", dist.Uniform(0., 1.))

    theta_blg_ps = numpyro.sample("theta_bulge_ps", dist.Dirichlet(jnp.ones((self.n_blg,)) / self.n_blg))
    temp_blg_ps = jnp.sum(theta_blg_ps[:, None] * self.blg_s, 0)

    temp_nfw_ps = self.nfw_temp_gen.get_NFW2_template(gamma=numpyro.sample("gamma_ps", dist.Uniform(0.2, 2.)))
    temp_nfw_ps /= jnp.mean(temp_nfw_ps[~self.nm])

    temp_gce_ps = f_bulge_ps * temp_blg_ps + (1 - f_bulge_ps) * temp_nfw_ps

    #=== point source: disk ===
    Sps_dsk = numpyro.sample("Sps_dsk", dist.Uniform(1e-5, 4.))
    zs = numpyro.sample("zs", dist.Uniform(0.1, 2.5))
    C = numpyro.sample("C", dist.Uniform(0.05, 8.))
    temp_dsk_ps = self.dsk_temp_gen.get_template(zs=zs, C=C)
    temp_dsk_ps /= jnp.mean(temp_dsk_ps[~self.nm])
    
    #=== point source: source count function and normalization ===
    Sps_list = [Sps_gce, Sps_dsk]
    npt_compressed = jnp.array([temp_gce_ps, temp_dsk_ps])
    theta = []
    s_arr = jnp.logspace(-1., 2., 1000)
    for i, ps in enumerate(["gce", "dsk"]):
        n1 = numpyro.sample(f'n1_{ps}', dist.Uniform(2.1, 8))
        n2 = numpyro.sample(f'n2_{ps}', dist.Uniform(0.5, 2))
        n3 = numpyro.sample(f'n3_{ps}', dist.Uniform(-8, -1))
        sb1 = numpyro.sample(f'sb1_{ps}', dist.Uniform(5., 40.))
        lambda_s = numpyro.sample(f'lambdas_{ps}', dist.Uniform(0.1, 0.95))

        theta_tmp = jnp.array([1., n1, n2, n3, sb1, lambda_s * sb1])
        dnds_arr = dnds(s_arr, theta_tmp)
        A = Sps_list[i] / jnp_trapezoid(s_arr * dnds_arr, s_arr)
        theta.append([A, n1, n2, n3, sb1, lambda_s * sb1])
    theta = jnp.array(theta)
    
    #=== point source: adjust with exposure regions ===
    # pad the last exposure region so that all are the same size
    exp_lens = [len(self.expreg_indices[i]) for i in range(len(self.expreg_indices))]
    n_pad = exp_lens[0] - exp_lens[-1]
    
    expreg_indices = jnp.zeros_like(self.expreg_indices)
    expreg_indices = expreg_indices.at[:-1].set(self.expreg_indices[:-1])
    expreg_indices = expreg_indices.at[-1].set(jnp.pad(self.expreg_indices[-1], (0, n_pad)))

    log_like_np_exp_vmapped = jax.vmap(log_like_np, in_axes=(0, 0, 1, 0, None, None, None, None))
            
    mu_batch = mu[~self.mask_roi][jnp.array(expreg_indices)]
    npt_compressed_batch = npt_compressed[:, ~self.mask_roi][:, jnp.array(expreg_indices)]
    data_batch = data[~self.mask_roi][jnp.array(expreg_indices)]
    exposure_multiplier = self.exposure_means_list / self.exposure_mean
    
    # scale non-Poissonian parameters (norm divided by exposure ratio, breaks multiplied)
    theta = repeat(theta, "n_ps n_param -> n_exp n_ps n_param", n_exp=len(expreg_indices))
    theta = theta.at[:, :, 0].set(theta[:, :, 0] / exposure_multiplier[:, None])
    theta = theta.at[:, :, -1].set(theta[:, :, -1] * exposure_multiplier[:, None])
    theta = theta.at[:, :, -2].set(theta[:, :, -2] * exposure_multiplier[:, None])

    #=== compute likelihood ===
    with numpyro.plate("data", size=len(mu[~self.mask_roi]), dim=-1):
        
        log_like_exp = log_like_np_exp_vmapped(
            theta,
            mu_batch,
            npt_compressed_batch,
            data_batch,
            self.f_ary,
            self.df_rho_ary,
            self.k_max,
            len(expreg_indices[0])
        )
        loglike = jnp.concatenate(log_like_exp)[:len(mu[~self.mask_roi])]
        print(loglike)

        return loglike

        with handlers.mask(mask=~jnp.logical_or(jnp.isinf(loglike), jnp.isnan(loglike))):
            with handlers.scale(scale=beta): # optional tempering the likelihood for SVI with beta schedule
                return numpyro.factor('log-likelihood', loglike)